In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FPFSShearEstimator(nn.Module):
    def __init__(self, smoothing_scale=1.0, order=2):
        super().__init__()
        self.smoothing_scale = smoothing_scale
        self.order = order
        
        # Learnable PSF model
        self.psf_model = nn.Parameter(torch.randn(1, 1, 32, 32))
        
        # Pre-compute factorials as tensors for differentiability
        max_n = self.order * 2 + 1
        self._factorial_cache = self._precompute_factorials(max_n)
        
        # Dictionary to store mode mapping
        self.mode_map = {}
        
    def _precompute_factorials(self, max_n):
        """Precompute factorials as PyTorch tensors"""
        factorials = torch.ones(max_n)
        for i in range(1, len(factorials)):
            factorials[i] = factorials[i-1] * i
        return factorials
    
    def _factorial(self, n):
        """Get factorial from precomputed cache"""
        if n < len(self._factorial_cache):
            return self._factorial_cache[n]
        else:
            # Fallback for larger values (should not happen with reasonable orders)
            result = self._factorial_cache[-1]
            for i in range(len(self._factorial_cache), n+1):
                result = result * i
            return result
    
    def generate_shapelet_basis(self, size):
        """
        Generate differentiable shapelet basis functions using only PyTorch operations
        """
        # Create coordinates grid
        x = torch.linspace(-1, 1, size, device=self.psf_model.device)
        y = torch.linspace(-1, 1, size, device=self.psf_model.device)
        X, Y = torch.meshgrid(x, y, indexing='ij')
        
        # Polar coordinates
        R = torch.sqrt(X**2 + Y**2)
        Theta = torch.atan2(Y, X)
        
        # Generate shapelet modes
        shapelet_modes = []
        self.mode_map = {}  # Reset mode map
        mode_idx = 0
        
        for n in range(self.order + 1):
            for m in range(-n, n+1):
                # Only include modes where n and m have the same parity
                if (n - abs(m)) % 2 == 0:
                    # Use differentiable function for shapelet calculation
                    mode = self._shapelet_basis_function(n, m, R, Theta)
                    shapelet_modes.append(mode.unsqueeze(0))
                    
                    # Track mode indices in dictionary
                    self.mode_map[(n, m)] = mode_idx
                    mode_idx += 1
        
        # Stack all modes into a single tensor
        return torch.cat(shapelet_modes, dim=0)
    
    def _laguerre_polynomial(self, n, alpha, x):
        """
        Compute generalized Laguerre polynomial using stable recurrence relation.
        All operations are in PyTorch for differentiability.
        """
        if n == 0:
            return torch.ones_like(x)
        elif n == 1:
            return 1 + alpha - x
        else:
            # Use recurrence relation with PyTorch operations
            L_prev = torch.ones_like(x)
            L_curr = 1 + alpha - x
            
            for k in range(2, n+1):
                L_next = ((2*k - 1 + alpha - x) * L_curr - (k - 1 + alpha) * L_prev) / k
                L_prev, L_curr = L_curr, L_next
                
            return L_curr
    
    def _shapelet_basis_function(self, n, m, R, Theta):
        """
        Compute a single shapelet basis function using PyTorch operations.
        """
        # Use torch.tensor for scalar values
        abs_m = abs(m)
        n_factorial = self._factorial(n).to(R.device)
        n_abs_m_factorial = self._factorial(n + abs_m).to(R.device)
        
        # Compute normalization constant with PyTorch
        norm_const = torch.sqrt(
            n_factorial / 
            (torch.tensor(torch.pi).to(R.device) * n_abs_m_factorial)
        )
        
        # Squared radius scaled by smoothing parameter
        r_squared = R**2 / self.smoothing_scale**2
        
        # Radial part using generalized Laguerre polynomial
        radial_part = self._laguerre_polynomial(
            (n - abs_m) // 2,  # Integer division for proper index
            abs_m,
            r_squared
        )
        
        # Angular part depends on sign of m
        if m >= 0:
            angular_part = torch.cos(m * Theta)
        else:  # m < 0
            angular_part = torch.sin(abs_m * Theta)
        
        # Full shapelet function with proper scaling
        shapelet = norm_const * (R / self.smoothing_scale)**abs_m * \
                  torch.exp(-r_squared / 2) * \
                  radial_part * angular_part
        
        return shapelet
    
    def fourier_shapelet_transform(self, image):
        """
        Compute Fourier-space shapelet transform using PyTorch FFT
        """
        # Ensure proper tensor shapes
        batch_size, channels, height, width = image.shape
        
        # Resize PSF to match image size
        if self.psf_model.shape[-1] != width or self.psf_model.shape[-2] != height:
            psf = F.interpolate(
                self.psf_model, 
                size=(height, width), 
                mode='bilinear', 
                align_corners=False
            )
        else:
            psf = self.psf_model
        
        # Expand PSF to match batch size and channels
        psf = psf.expand(batch_size, channels, -1, -1)
        
        # Perform forward FFTs
        image_fft = torch.fft.rfft2(image)
        psf_fft = torch.fft.rfft2(psf)
        
        # Deconvolution in Fourier space with regularization
        eps = 1e-6
        deconvolved_fft = image_fft / (psf_fft + eps)
        
        # Inverse FFT to get back to image space
        deconvolved_image = torch.fft.irfft2(deconvolved_fft, s=(height, width))
        
        # Generate shapelet basis
        shapelet_basis = self.generate_shapelet_basis(width)
        num_modes = shapelet_basis.shape[0]
        
        # Fix for batch matrix multiplication
        # Reshape tensors properly for projection
        basis_flat = shapelet_basis.reshape(num_modes, -1)  # [num_modes, height*width]
        
        # Process each batch and channel separately to avoid dimension issues
        all_moments = []
        for b in range(batch_size):
            channel_moments = []
            for c in range(channels):
                # Get current image
                img = deconvolved_image[b, c].reshape(-1)  # [height*width]
                
                # Project onto basis
                moments = torch.matmul(basis_flat, img)  # [num_modes]
                channel_moments.append(moments.unsqueeze(0))
            
            # Stack channels
            batch_moments = torch.cat(channel_moments, dim=0)  # [channels, num_modes]
            all_moments.append(batch_moments.unsqueeze(0))
        
        # Stack batches
        shapelet_moments = torch.cat(all_moments, dim=0)  # [batch_size, channels, num_modes]
        
        return shapelet_moments
    
    def compute_ellipticity(self, shapelet_moments):
        """
        Compute ellipticity from shapelet moments
        """
        # Get indices for the required moments
        idx_00 = self.mode_map.get((0, 0), 0)  # Monopole
        idx_22 = self.mode_map.get((2, 2), 1)  # Real part of quadrupole
        idx_2m2 = self.mode_map.get((2, -2), 2)  # Imaginary part of quadrupole
        
        # Extract moments safely
        batch_size, channels, _ = shapelet_moments.shape
        
        # Safe indexing
        M00 = shapelet_moments[:, :, idx_00]
        M22_real = shapelet_moments[:, :, idx_22]
        M22_imag = shapelet_moments[:, :, idx_2m2]
        
        # Compute complex ellipticity with stabilization
        eps = 1e-6
        ellipticity_real = M22_real / (M00 + eps)
        ellipticity_imag = M22_imag / (M00 + eps)
        
        # Create complex tensor for ellipticity
        ellipticity = torch.complex(ellipticity_real, ellipticity_imag)
        
        return ellipticity
    
    def forward(self, galaxy_images):
        """
        Main forward pass for shear estimation
        """
        # Ensure input has 4 dimensions: [batch, channels, height, width]
        if len(galaxy_images.shape) == 3:
            galaxy_images = galaxy_images.unsqueeze(1)
        
        # Batch processing of galaxy images
        shapelet_moments = self.fourier_shapelet_transform(galaxy_images)
        
        # Compute ellipticity for each galaxy
        ellipticities = self.compute_ellipticity(shapelet_moments)
        
        # Average ellipticities to estimate shear
        # Reduced shear g is approximately e/2 in weak lensing
        shear_estimate = torch.mean(ellipticities, dim=0)
        
        # Apply responsivity correction - typically a factor of 2 in weak lensing
        return shear_estimate / 2.0


def test_fpfs_shear_estimator():
    """Test the differentiable FPFS pipeline with gradients"""
    # Set up a small test
    batch_size = 10
    image_size = 64
    
    # Create random galaxy images
    galaxy_images = torch.randn(batch_size, 1, image_size, image_size, requires_grad=True)
    print(galaxy_images.shape)
    
    # Initialize FPFS Shear Estimator
    shear_estimator = FPFSShearEstimator(order=4)
    
    # Compute shear
    try:
        shear = shear_estimator(galaxy_images)
        print("Estimated Shear:", shear)
        
        # Test differentiability by computing gradients
        loss = torch.abs(shear).sum()
        loss.backward()
        
        # Check if gradients propagated
        if galaxy_images.grad is not None:
            print("Pipeline is differentiable! Gradient norm:", galaxy_images.grad.norm().item())
        else:
            print("Error: Pipeline is not differentiable")
        
        # Check if PSF model received gradients
        if shear_estimator.psf_model.grad is not None:
            print("PSF model received gradients! Norm:", shear_estimator.psf_model.grad.norm().item())
        else:
            print("Error: PSF model did not receive gradients")
            
    except Exception as e:
        print(f"Error during testing: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    test_fpfs_shear_estimator()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FPFSShearEstimator(nn.Module):
    def __init__(self, smoothing_scale=1.0, order=2):
        super().__init__()
        self.smoothing_scale = smoothing_scale
        self.order = order
        
        # Learnable PSF model
        self.psf_model = nn.Parameter(torch.randn(1, 1, 32, 32))
        
        # Pre-compute factorials as tensors for differentiability
        max_n = self.order * 2 + 1
        self._factorial_cache = self._precompute_factorials(max_n)
        
        # Dictionary to store mode mapping
        self.mode_map = {}
        
    def _precompute_factorials(self, max_n):
        """Precompute factorials as PyTorch tensors"""
        factorials = torch.ones(max_n)
        for i in range(1, len(factorials)):
            factorials[i] = factorials[i-1] * i
        return factorials
    
    def _factorial(self, n):
        """Get factorial from precomputed cache"""
        if n < len(self._factorial_cache):
            return self._factorial_cache[n]
        else:
            # Fallback for larger values (should not happen with reasonable orders)
            result = self._factorial_cache[-1]
            for i in range(len(self._factorial_cache), n+1):
                result = result * i
            return result
    
    def generate_shapelet_basis(self, size):
        """
        Generate differentiable shapelet basis functions using only PyTorch operations
        """
        # Create coordinates grid
        x = torch.linspace(-1, 1, size, device=self.psf_model.device)
        y = torch.linspace(-1, 1, size, device=self.psf_model.device)
        X, Y = torch.meshgrid(x, y, indexing='ij')
        
        # Polar coordinates
        R = torch.sqrt(X**2 + Y**2)
        Theta = torch.atan2(Y, X)
        
        # Generate shapelet modes
        shapelet_modes = []
        self.mode_map = {}  # Reset mode map
        mode_idx = 0
        
        for n in range(self.order + 1):
            for m in range(-n, n+1):
                # Only include modes where n and m have the same parity
                if (n - abs(m)) % 2 == 0:
                    # Use differentiable function for shapelet calculation
                    mode = self._shapelet_basis_function(n, m, R, Theta)
                    shapelet_modes.append(mode.unsqueeze(0))
                    
                    # Track mode indices in dictionary
                    self.mode_map[(n, m)] = mode_idx
                    mode_idx += 1
        
        # Stack all modes into a single tensor
        return torch.cat(shapelet_modes, dim=0)
    
    def _laguerre_polynomial(self, n, alpha, x):
        """
        Compute generalized Laguerre polynomial using stable recurrence relation.
        All operations are in PyTorch for differentiability.
        """
        if n == 0:
            return torch.ones_like(x)
        elif n == 1:
            return 1 + alpha - x
        else:
            # Use recurrence relation with PyTorch operations
            L_prev = torch.ones_like(x)
            L_curr = 1 + alpha - x
            
            for k in range(2, n+1):
                L_next = ((2*k - 1 + alpha - x) * L_curr - (k - 1 + alpha) * L_prev) / k
                L_prev, L_curr = L_curr, L_next
                
            return L_curr
    
    def _shapelet_basis_function(self, n, m, R, Theta):
        """
        Compute a single shapelet basis function using PyTorch operations.
        """
        # Use torch.tensor for scalar values
        abs_m = abs(m)
        n_factorial = self._factorial(n).to(R.device)
        n_abs_m_factorial = self._factorial(n + abs_m).to(R.device)
        
        # Compute normalization constant with PyTorch
        norm_const = torch.sqrt(
            n_factorial / 
            (torch.tensor(torch.pi).to(R.device) * n_abs_m_factorial)
        )
        
        # Squared radius scaled by smoothing parameter
        r_squared = R**2 / self.smoothing_scale**2
        
        # Radial part using generalized Laguerre polynomial
        radial_part = self._laguerre_polynomial(
            (n - abs_m) // 2,  # Integer division for proper index
            abs_m,
            r_squared
        )
        
        # Angular part depends on sign of m
        if m >= 0:
            angular_part = torch.cos(m * Theta)
        else:  # m < 0
            angular_part = torch.sin(abs_m * Theta)
        
        # Full shapelet function with proper scaling
        shapelet = norm_const * (R / self.smoothing_scale)**abs_m * \
                  torch.exp(-r_squared / 2) * \
                  radial_part * angular_part
        
        return shapelet
    
    def fourier_shapelet_transform(self, image):
        """
        Compute Fourier-space shapelet transform using PyTorch FFT
        """
        # Ensure proper tensor shapes
        batch_size, channels, height, width = image.shape
        
        # Resize PSF to match image size
        if self.psf_model.shape[-1] != width or self.psf_model.shape[-2] != height:
            psf = F.interpolate(
                self.psf_model, 
                size=(height, width), 
                mode='bilinear', 
                align_corners=False
            )
        else:
            psf = self.psf_model
        
        # Expand PSF to match batch size and channels
        psf = psf.expand(batch_size, channels, -1, -1)
        
        # Perform forward FFTs
        image_fft = torch.fft.rfft2(image)
        psf_fft = torch.fft.rfft2(psf)
        
        # Deconvolution in Fourier space with regularization
        eps = 1e-6
        deconvolved_fft = image_fft / (psf_fft + eps)
        
        # Inverse FFT to get back to image space
        deconvolved_image = torch.fft.irfft2(deconvolved_fft, s=(height, width))
        
        # Generate shapelet basis
        shapelet_basis = self.generate_shapelet_basis(width)
        num_modes = shapelet_basis.shape[0]
        
        # Fix for batch matrix multiplication
        # Reshape tensors properly for projection
        basis_flat = shapelet_basis.reshape(num_modes, -1)  # [num_modes, height*width]
        
        # Process each batch and channel separately to avoid dimension issues
        all_moments = []
        for b in range(batch_size):
            channel_moments = []
            for c in range(channels):
                # Get current image
                img = deconvolved_image[b, c].reshape(-1)  # [height*width]
                
                # Project onto basis
                moments = torch.matmul(basis_flat, img)  # [num_modes]
                channel_moments.append(moments.unsqueeze(0))
            
            # Stack channels
            batch_moments = torch.cat(channel_moments, dim=0)  # [channels, num_modes]
            all_moments.append(batch_moments.unsqueeze(0))
        
        # Stack batches
        shapelet_moments = torch.cat(all_moments, dim=0)  # [batch_size, channels, num_modes]
        
        return shapelet_moments
    
    def compute_ellipticity(self, shapelet_moments):
        # Get indices for required moments
        idx_00 = self.mode_map[(0, 0)]
        idx_22 = self.mode_map[(2, 2)]
        idx_2m2 = self.mode_map[(2, -2)]

        M00 = shapelet_moments[..., idx_00]
        M22_real = shapelet_moments[..., idx_22]
        M22_imag = shapelet_moments[..., idx_2m2]

        eps = 1e-6
        e1 = M22_real / (M00 + eps)
        e2 = M22_imag / (M00 + eps)
        
        return torch.stack([e1, e2], dim=-1)  # Real tensor with last dim=2
    
    def forward(self, galaxy_images):
        shapelet_moments = self.fourier_shapelet_transform(galaxy_images)
        ellipticities = self.compute_ellipticity(shapelet_moments)
        shear_estimate = torch.mean(ellipticities, dim=0)
        return shear_estimate / 2.0  # Returns shape [channels, 2]


def test_fpfs_shear_estimator():
    """Test the differentiable FPFS pipeline with gradients"""
    # Set up a small test
    batch_size = 10
    image_size = 64
    
    # Create random galaxy images
    galaxy_images = torch.randn(batch_size, 1, image_size, image_size, requires_grad=True)
    print(galaxy_images.shape)
    
    # Initialize FPFS Shear Estimator
    shear_estimator = FPFSShearEstimator(order=4)
    
    # Compute shear
    try:
        shear = shear_estimator(galaxy_images)
        print("Estimated Shear:", shear)
        
        # Test differentiability by computing gradients
        loss = torch.abs(shear).sum()
        loss.backward()
        
        # Check if gradients propagated
        if galaxy_images.grad is not None:
            print("Pipeline is differentiable! Gradient norm:", galaxy_images.grad.norm().item())
        else:
            print("Error: Pipeline is not differentiable")
        
        # Check if PSF model received gradients
        if shear_estimator.psf_model.grad is not None:
            print("PSF model received gradients! Norm:", shear_estimator.psf_model.grad.norm().item())
        else:
            print("Error: PSF model did not receive gradients")
            
    except Exception as e:
        print(f"Error during testing: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    test_fpfs_shear_estimator()

torch.Size([10, 1, 64, 64])
Estimated Shear: tensor([[0.1577, 0.4904]], grad_fn=<DivBackward0>)
Pipeline is differentiable! Gradient norm: 0.8732166886329651
PSF model received gradients! Norm: 5.453855514526367


In [3]:
gt = torch.load("gt.pth")
gt = gt.unsqueeze(0).unsqueeze(0)
gt.shape
shear_estimator = FPFSShearEstimator(order=4)
shear = shear_estimator(gt)
shear

/var/folders/3z/xzj6jd1x4d9cy2w36g2v23b00000gn/T/ipykernel_32793/1084892399.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gt = torch.load("gt.pth")


tensor([[-0.0517, -0.0396]], grad_fn=<DivBackward0>)